# Dash Components: Flight Delay Time Statistics Dashboard

**CISC7204 Assignment 02 – Module 04 Lab 04**

## Objectives
- Know how to add multiple graphs to the dashboard
- Work with Dash Callbacks to handle multiple outputs

**Estimated time needed:** 30 minutes

## Dataset Used
[Airline Reporting Carrier On-Time Performance dataset](airline_data.csv) (local file, same directory as this notebook)

## Theme
Analyze flight delays in a dashboard.

## Dashboard Components
- Monthly average carrier delay by reporting airline for the given year
- Monthly average weather delay by reporting airline for the given year
- Monthly average national air system delay by reporting airline for the given year
- Monthly average security delay by reporting airline for the given year
- Monthly average late aircraft delay by reporting airline for the given year

> NOTE: Year range should be between 2010 and 2020

## Task 1 – Read the data

Import the required libraries and read the dataset from the local `airline_data.csv` file.

In [1]:
# Import required libraries
import pandas as pd
import plotly.graph_objects as go
import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output
import plotly.express as px

# Read the airline data into pandas dataframe (local file)
airline_data = pd.read_csv('airline_data.csv',
                           encoding = "ISO-8859-1",
                           dtype={'Div1Airport': str, 'Div1TailNum': str,
                                  'Div2Airport': str, 'Div2TailNum': str})
print('Data loaded. Shape:', airline_data.shape)

Data loaded. Shape: (27000, 110)


## Task 2 & 3 – Create the Dash application and update layout components

The dashboard layout contains:
- Application title: `Flight Delay Time Statistics`, center aligned, color `#503D36`, font-size 30
- Input component: id `input-year`, default value `2010`, type `number`
- Segment 1: `carrier-plot` and `weather-plot` side by side
- Segment 2: `nas-plot` and `security-plot` side by side
- Segment 3: `late-plot` (width 65%)

In [2]:
# Create a dash application
app = dash.Dash(__name__)

# Build dash app layout
app.layout = html.Div(children=[html.H1('Flight Delay Time Statistics',
                                style={'textAlign': 'center', 'color': '#503D36',
                                'font-size': 30}),
                                html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                type='number', style={'height':'35px', 'font-size': 30}),],
                                style={'font-size': 30}),
                                html.Br(),
                                html.Br(),
                                # Segment 1
                                html.Div([
                                        html.Div(dcc.Graph(id='carrier-plot')),
                                        html.Div(dcc.Graph(id='weather-plot'))
                                ], style={'display': 'flex'}),
                                # Segment 2
                                html.Div([
                                        html.Div(dcc.Graph(id='nas-plot')),
                                        html.Div(dcc.Graph(id='security-plot'))
                                ], style={'display': 'flex'}),
                                # Segment 3
                                html.Div(dcc.Graph(id='late-plot'), style={'width':'65%'})
                                ])
print('Dash app layout created.')

Dash app layout created.


## Task 4 – Supporting helper function

`compute_info` takes the airline data and a selected year, and returns the computed average dataframes for the five delay types.

In [3]:
"""Compute_info function description

This function takes in airline data and selected year as an input and performs computation for creating charts and plots.

Arguments:
    airline_data: Input airline data.
    entered_year: Input year for which computation needs to be performed.

Returns:
    Computed average dataframes for carrier delay, weather delay, NAS delay, security delay, and late aircraft delay.
"""
def compute_info(airline_data, entered_year):
    # Select data
    df = airline_data[airline_data['Year']==int(entered_year)]
    # Compute delay averages
    avg_car = df.groupby(['Month','Reporting_Airline'])['CarrierDelay'].mean().reset_index()
    avg_weather = df.groupby(['Month','Reporting_Airline'])['WeatherDelay'].mean().reset_index()
    avg_NAS = df.groupby(['Month','Reporting_Airline'])['NASDelay'].mean().reset_index()
    avg_sec = df.groupby(['Month','Reporting_Airline'])['SecurityDelay'].mean().reset_index()
    avg_late = df.groupby(['Month','Reporting_Airline'])['LateAircraftDelay'].mean().reset_index()
    return avg_car, avg_weather, avg_NAS, avg_sec, avg_late

## Task 5 & 6 – Callback function returning five figures

The callback takes the entered year, computes the five delay averages through `compute_info`, and returns the five line plots.

In [4]:
# Callback decorator
@app.callback([
               Output(component_id='carrier-plot', component_property='figure'),
               Output(component_id='weather-plot', component_property='figure'),
               Output(component_id='nas-plot', component_property='figure'),
               Output(component_id='security-plot', component_property='figure'),
               Output(component_id='late-plot', component_property='figure')
               ],
              Input(component_id='input-year', component_property='value'))

# Computation to callback function and return graph
def get_graph(entered_year):
    # Compute required information for creating graph from the data
    avg_car, avg_weather, avg_NAS, avg_sec, avg_late = compute_info(airline_data, entered_year)

    # Line plot for carrier delay
    carrier_fig = px.line(avg_car, x='Month', y='CarrierDelay', color='Reporting_Airline', title='Average carrier delay time (minutes) by airline')
    # Line plot for weather delay
    weather_fig = px.line(avg_weather, x='Month', y='WeatherDelay', color='Reporting_Airline', title='Average weather delay time (minutes) by airline')
    # Line plot for nas delay
    nas_fig = px.line(avg_NAS, x='Month', y='NASDelay', color='Reporting_Airline', title='Average NAS delay time (minutes) by airline')
    # Line plot for security delay
    sec_fig = px.line(avg_sec, x='Month', y='SecurityDelay', color='Reporting_Airline', title='Average security delay time (minutes) by airline')
    # Line plot for late aircraft delay
    late_fig = px.line(avg_late, x='Month', y='LateAircraftDelay', color='Reporting_Airline', title='Average late aircraft delay time (minutes) by airline')

    return [carrier_fig, weather_fig, nas_fig, sec_fig, late_fig]

### Demo: invoke the callback logic directly (2010) to show all five charts in this notebook

In [5]:
figs = get_graph(2010)
for f in figs:
    f.show()

## Run the application

The application can be run as a standalone script (e.g. `python flight_delay.py`). Inside this notebook the server is **not** started to avoid blocking the kernel.

```python
if __name__ == '__main__':
    app.run()
```

In [6]:
# Run the app (commented out in the notebook to avoid blocking the kernel)
# if __name__ == '__main__':
#     app.run()

## Exercise – Practice Task

Change the dashboard title from `Flight Delay Time Statistics` to `Flight Details Statistics Dashboard` with font-size 35.

In [7]:
# Practice exercise: updated title with font-size 35
app2 = dash.Dash(__name__)
app2.layout = html.Div(children=[html.H1('Flight Details Statistics Dashboard',
                                style={'textAlign': 'center', 'color': '#503D36',
                                'font-size': 35}),
                                html.Div(["Input Year: ", dcc.Input(id='input-year', value='2010',
                                type='number', style={'height':'35px', 'font-size': 30}),],
                                style={'font-size': 30}),
                                html.Br(),
                                html.Br(),
                                html.Div([
                                        html.Div(dcc.Graph(id='carrier-plot')),
                                        html.Div(dcc.Graph(id='weather-plot'))
                                ], style={'display': 'flex'}),
                                html.Div([
                                        html.Div(dcc.Graph(id='nas-plot')),
                                        html.Div(dcc.Graph(id='security-plot'))
                                ], style={'display': 'flex'}),
                                html.Div(dcc.Graph(id='late-plot'), style={'width':'65%'})
                                ])
print('Updated dashboard layout with new title created.')

Updated dashboard layout with new title created.


### Summary
- Added multiple graphs to a single dashboard
- Used a single callback with multiple outputs (five delay charts)
- To stop a running app in the terminal, press `ctrl+c`